# Telco Customer Churn — Baseline Modeling

Цель ноутбука:
- обучить и сравнить базовые модели для задачи churn prediction;
- корректно оценить качество на валидационной и тестовой выборках;
- зафиксировать сильный интерпретируемый baseline перед переходом к более сложным моделям.

Модели:
- Logistic Regression (интерпретируемый baseline);
- Random Forest (нелинейный baseline).

Метрики:
- ROC-AUC;
- PR-AUC;
- Precision / Recall / F1 для класса churn (1).


In [23]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report,
)

RANDOM_STATE = 42

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)


In [24]:
TARGET_COL = "churn_value"

train = pd.read_csv("../data/processed/train.csv")
valid = pd.read_csv("../data/processed/valid.csv")
test  = pd.read_csv("../data/processed/test.csv")

X_train = train.drop(columns=[TARGET_COL])
y_train = train[TARGET_COL]

X_valid = valid.drop(columns=[TARGET_COL])
y_valid = valid[TARGET_COL]

X_test  = test.drop(columns=[TARGET_COL])
y_test  = test[TARGET_COL]

X_train.shape, X_valid.shape, X_test.shape


((4225, 24), (1409, 24), (1409, 24))

Используем уже подготовленные выборки из `02_feature_engineering.ipynb`:

- признаки очищены и дополнены бизнес-фичами;
- из данных удалены ID, география и признаки с утечкой таргета;
- разбиение на `train / valid / test` сделано со стратификацией по `churn_value`.


In [25]:
# Числовые и категориальные признаки для пайплайнов
num_features = X_train.select_dtypes(include=["number"]).columns.tolist()
cat_features = X_train.select_dtypes(include=["object", "string", "category"]).columns.tolist()

num_features, cat_features


(['tenure_months',
  'monthly_charges',
  'total_charges',
  'is_new_customer',
  'is_long_contract',
  'has_internet',
  'num_addon_services',
  'avg_monthly_revenue'],
 ['gender',
  'senior_citizen',
  'partner',
  'dependents',
  'phone_service',
  'multiple_lines',
  'internet_service',
  'online_security',
  'online_backup',
  'device_protection',
  'tech_support',
  'streaming_tv',
  'streaming_movies',
  'contract',
  'paperless_billing',
  'payment_method'])

Для базовых моделей используем следующий препроцессинг:

- Logistic Regression:
  - стандартизация числовых признаков (`StandardScaler`);
  - one-hot кодирование категориальных признаков (`OneHotEncoder`).
- Random Forest:
  - числовые признаки передаются как есть (`passthrough`);
  - категориальные — через one-hot кодирование.

OneHotEncoder используется с `handle_unknown="ignore"` и `sparse_output=False`
(актуальный параметр для новых версий scikit-learn).


In [26]:
def evaluate_model(model, X, y, name="model"):
    """
    Оценивает модель по ROC-AUC и PR-AUC (Average Precision)
    и возвращает словарь с результатами.
    """
    proba = model.predict_proba(X)[:, 1]

    roc = roc_auc_score(y, proba)
    pr  = average_precision_score(y, proba)

    print(f"{name:20s} | ROC-AUC: {roc:.4f} | PR-AUC: {pr:.4f}")
    return {"model": name, "roc_auc": roc, "pr_auc": pr}


## Logistic Regression — интерпретируемый baseline


In [27]:
# Препроцессинг для Logistic Regression
preprocess_lr = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_features),
    ]
)

logreg = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)

pipe_lr = Pipeline(
    steps=[
        ("preprocess", preprocess_lr),
        ("model", logreg),
    ]
)

# Обучение
pipe_lr.fit(X_train, y_train)

# Оценка на train и valid
res_lr_train = evaluate_model(pipe_lr, X_train, y_train, "LogReg (train)")
res_lr_valid = evaluate_model(pipe_lr, X_valid, y_valid, "LogReg (valid)")

res_lr_train, res_lr_valid


LogReg (train)       | ROC-AUC: 0.8678 | PR-AUC: 0.6913
LogReg (valid)       | ROC-AUC: 0.8530 | PR-AUC: 0.6701


({'model': 'LogReg (train)',
  'roc_auc': 0.8678250043683383,
  'pr_auc': 0.6913450366298856},
 {'model': 'LogReg (valid)',
  'roc_auc': 0.8530277196517606,
  'pr_auc': 0.6701397678457769})

Logistic Regression используется как интерпретируемый baseline:

- `class_weight="balanced"` учитывает дисбаланс классов (≈27% churn);
- стандартизация численных признаков позволяет корректно обучать линейную модель;
- one-hot кодирование категорий даёт простую интерпретацию важности признаков
  (через веса модели и последующий анализ коэффициентов).


## Random Forest — нелинейный baseline


In [28]:
# Препроцессинг для Random Forest
preprocess_rf = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_features),
    ]
)

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=10,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    class_weight="balanced",
)

pipe_rf = Pipeline(
    steps=[
        ("preprocess", preprocess_rf),
        ("model", rf),
    ]
)

# Обучение
pipe_rf.fit(X_train, y_train)

# Оценка на train и valid
res_rf_train = evaluate_model(pipe_rf, X_train, y_train, "RF (train)")
res_rf_valid = evaluate_model(pipe_rf, X_valid, y_valid, "RF (valid)")

res_rf_train, res_rf_valid


RF (train)           | ROC-AUC: 0.9177 | PR-AUC: 0.8016
RF (valid)           | ROC-AUC: 0.8528 | PR-AUC: 0.6611


({'model': 'RF (train)',
  'roc_auc': 0.9177030932433302,
  'pr_auc': 0.8015569163170303},
 {'model': 'RF (valid)',
  'roc_auc': 0.8527900488258544,
  'pr_auc': 0.6611258404909556})

Random Forest используется как нелинейный baseline:

- учитывает взаимодействия признаков и нелинейные зависимости;
- параметр `min_samples_leaf=10` уменьшает переобучение;
- `class_weight="balanced"` компенсирует дисбаланс классов.


## Сравнение baseline-моделей на validation


In [29]:
results = pd.DataFrame([res_lr_valid, res_rf_valid]).sort_values("roc_auc", ascending=False)
results


,model,roc_auc,pr_auc
0,LogReg (valid),0.8530,0.6701
1,RF (valid),0.8528,0.6611


### Сравнение baseline-моделей на validation

По результатам на валидационной выборке:

- **Logistic Regression** показывает чуть более высокие ROC-AUC и PR-AUC
  (0.8530 и 0.6701), чем **Random Forest** (0.8528 и 0.6611).
- Разница небольшая, но при этом Random Forest сильнее переобучается
  (ROC-AUC на train существенно выше, чем на valid).

Поэтому в роли основного интерпретируемого baseline в дальнейшем
будем использовать **Logistic Regression**, а Random Forest рассматривать
как альтернативный нелинейный baseline.


## Precision / Recall / F1 по классу churn (validation)


In [ ]:
# выбираем лучшую модель по валидации
print("LogReg (valid):")
y_valid_pred_lr = pipe_lr.predict(X_valid)
print(classification_report(y_valid, y_valid_pred_lr, digits=4))

print("\nRandom Forest (valid):")
y_valid_pred_rf = pipe_rf.predict(X_valid)
print(classification_report(y_valid, y_valid_pred_rf, digits=4))


LogReg (valid):
              precision    recall  f1-score   support

           0     0.9085    0.7575    0.8261      1035
           1     0.5403    0.7888    0.6413       374

    accuracy                         0.7658      1409
   macro avg     0.7244    0.7731    0.7337      1409
weighted avg     0.8107    0.7658    0.7771      1409


Random Forest (valid):
              precision    recall  f1-score   support

           0     0.9036    0.7700    0.8315      1035
           1     0.5484    0.7727    0.6415       374

    accuracy                         0.7708      1409
   macro avg     0.7260    0.7714    0.7365      1409
weighted avg     0.8093    0.7708    0.7811      1409



### Precision / Recall / F1 по классу churn (validation)

По отчёту видно, что обе модели показывают очень близкое качество:

- для класса `1` (churn) F1-меры почти совпадают (~0.64);
- Logistic Regression даёт чуть более высокий **recall** (чувствительность),
  а Random Forest — немного более высокую **precision**.

С учётом того, что LogReg чуть лучше по ROC-AUC / PR-AUC и менее переобучена,
её удобно использовать как основной интерпретируемый baseline.
Random Forest остаётся хорошей нелинейной альтернативой.


## Финальная оценка baseline-модели на test


In [37]:
evaluate_model(pipe_lr, X_test, y_test, "LogReg (test)")
evaluate_model(pipe_rf, X_test, y_test, "RF (test)")


LogReg (test)        | ROC-AUC: 0.8538 | PR-AUC: 0.6650
RF (test)            | ROC-AUC: 0.8528 | PR-AUC: 0.6619


{'model': 'RF (test)',
 'roc_auc': 0.8528068407863805,
 'pr_auc': 0.6619022991291692}

## Итоги baseline-моделирования

- Обучены две базовые модели: Logistic Regression и Random Forest.
- Обе модели показывают сопоставимое качество по ROC-AUC и PR-AUC
  на валидационной и тестовой выборках.
- **Logistic Regression**:
  - даёт чуть более высокие ROC-AUC и PR-AUC на valid и test;
  - меньше переобучается по сравнению с Random Forest;
  - остаётся интерпретируемым baseline, на который удобно опираться.
- **Random Forest**:
  - лучше описывает нелинейные зависимости;
  - даёт похожее качество, но сильнее переобучается.

В дальнейшем Logistic Regression используется как основной baseline
для сравнения с более продвинутыми моделями (CatBoost),
а Random Forest — как дополнительный нелинейный ориентир.
